https://docs.opendota.com/

In [26]:
import json
from dataclasses import dataclass
import datetime
import pathlib

import plotly.graph_objects as go
import requests

In [2]:
HEROES_FILE = pathlib.Path('heroes.json')

with open(HEROES_FILE) as f:
    HEROES_JSON = json.load(f)

HEROES = {int(heroidx): hero for heroidx, hero in HEROES_JSON.items()}

In [3]:
PLAYER_REQ = 'https://api.opendota.com/api/players/{account_id}'

def get_player(account_id):
    response = requests.get(PLAYER_REQ.format(account_id=account_id))
    if response.status_code != 200:
        return {}
    return response.json()


MATCHES_REQ = 'https://api.opendota.com/api/players/{account_id}/matches'

def get_matches(account_id):
    response = requests.get(MATCHES_REQ.format(account_id=account_id))
    if response.status_code != 200:
        return []
    return response.json()

In [4]:
AIRAT = 195123691
LION = 157739135
STAS = 1087711740

In [5]:
airat_stats = get_player(AIRAT)
lion_stats = get_player(LION)
stas_stats = get_player(STAS)

In [7]:
airat_matches = get_matches(AIRAT)

In [35]:
@dataclass
class Match:
    match_id: int
    start_time: datetime.datetime
    duration: datetime.timedelta
    win: bool
    is_radiant: bool
    hero_name: str
    kda: tuple[int, int, int]
    avg_rank: int

    @classmethod
    def from_json(cls, data):
        # player_slot: 0-127 are Radiant, 128-255 are Dire
        is_radiant = data['player_slot'] < 128
        win = data['radiant_win'] == is_radiant
        return cls(
            match_id=data['match_id'],
            start_time=datetime.datetime.fromtimestamp(data['start_time']),
            duration=datetime.timedelta(seconds=data['duration']),
            win=win,
            is_radiant=is_radiant,
            hero_name=HEROES[data['hero_id']]['localized_name'],
            kda=(data['kills'], data['deaths'], data['assists']),
            avg_rank=data['average_rank'],
        )
    
    def __str__(self):
        return f'({'-+'[self.win]})[{self.start_time:%H:%M %d.%m.%y}, {self.duration.seconds/60:.0f}min] {"/".join(map(str, self.kda))} "{self.hero_name}"'


In [51]:
class Analysis:
    def __init__(self, account_id):
        self.account_id = account_id
        self.player = get_player(account_id)
        matches_resp = get_matches(account_id)
        self.matches = [Match.from_json(match) for match in matches_resp if match['hero_id']]
    
    def get_match_duration_hist(self):
        durations = [match.duration.seconds for match in self.matches]
        fig = go.Figure(data=[go.Histogram(x=[d / 60 for d in durations], nbinsx=35)])
        fig.update_layout(title_text='Match duration histogram')
        fig.update_layout(template='plotly_dark')
        return fig
    
    def get_hero_statistic(self):
        hero_stats = {}
        for match in self.matches:
            hero_name = match.hero_name
            if hero_name not in hero_stats:
                hero_stats[hero_name] = {'played': 0, 'won': 0}
            hero_stats[hero_name]['played'] += 1
            if match.win:
                hero_stats[hero_name]['won'] += 1
        hero_stats = {hero_name: {'played': stats['played'], 'winrate': stats['won'] / stats['played']} for hero_name, stats in hero_stats.items()}
        hero_stats = {hero_name: stats for hero_name, stats in sorted(hero_stats.items(), key=lambda x: x[1]['played'], reverse=True)}
        hero_stats_short = {hero_name: stats for hero_name, stats in hero_stats.items() if stats['played'] > 9}
        fig = go.Figure(data=[go.Bar(x=list(hero_stats_short.keys()), y=[stats['played'] for stats in hero_stats_short.values()], marker_color=[stats['winrate'] for stats in hero_stats_short.values()])])
        fig.update_traces(hovertemplate='played: %{y}<br>winrate: %{marker.color:.1%}')
        fig.update_coloraxes(colorscale='Viridis', colorbar_title='Winrate')
        fig.update_layout(title_text='Hero statistics')
        fig.update_layout(template='plotly_dark')
        return fig

In [52]:
airat = Analysis(AIRAT)

In [46]:
airat.get_match_duration_hist().show()

In [53]:
airat.get_hero_statistic().show()